In [ ]:
!pip -q install transformers sentencepiece torch

## Import Libraries

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

## Load Pretrained Model

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.


## Response Generation Function

In [ ]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Tokenize the prompt
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)
    # Generate the model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()
    # Append assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

## Continuous Chat Loop

In [ ]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    # Initialize chat history with a system message to set behavior
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue
        # Generate reply from the model
        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        # fallback
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
Chatbot: Hello! How can I help you today?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans do. It encompasses various fields such as machine learning, natural language processing, computer vision, robotics, and more. The goal of AI is to create systems that can perform tasks that typically require human intelligence, such as recognizing speech or images, understanding natural language, or making decisions based on data analysis.
Chatbot: Python was created by Guido van Rossum. He began working on it in the late 1980s at CWI (Centrum Wiskunde & Informatica) in Amsterdam, Netherlands. The first version was released in 1991. Guido van Rossum has been the primary developer and active leader of the project ever since.
Chatbot: NLP stands for Natural Language Processing. It's a field of artificial intelligence focused on teaching

## 🧪 Sample Questions
- What is Machine Learning?
- Explain Python in simple words.
- Give interview tips for freshers.
- What is NLP?

In [1]:
import nbformat

fname = "Task_3_Chatbot_using_Hugging_Face_Transformers.ipynb"

nb = nbformat.read(fname, as_version=4)

# remove top-level widgets metadata
nb.metadata.pop("widgets", None)

# clean cell metadata and outputs
for cell in nb.cells:
    cell.metadata.pop("widgets", None)
    if "outputs" in cell:
        for out in cell["outputs"]:
            if "data" in out:
                out["data"].pop("application/vnd.jupyter.widget-view+json", None)
                out["data"].pop("application/vnd.jupyter.widget-state+json", None)

nbformat.write(nb, fname)
print("GitHub-safe notebook saved")

ModuleNotFoundError: No module named 'nbformat'